In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import sklearn
from mixed_data_pl_lime import LocalModelStabilityAnalyzer
import xgboost as xgb
df = pd.read_csv("adult.csv")

df = df.dropna()
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

feature_names = df.drop(columns='income').columns.tolist()
labels_raw = df['income'].values
data_raw = df.drop(columns='income').values.astype(str)  # Convert all values to strings for LabelEncoder

# 2. Label encoding
le = LabelEncoder()
labels = le.fit_transform(labels_raw)
class_names = le.classes_

# 3. Determine categorical variable indices (assuming non-numeric columns are categorical)
categorical_columns = [
    'workclass',
    'education',
    'marital-status',
    'occupation',
    'relationship',
    'race',
    'sex',
    'native-country'
]
n_features = len(feature_names)
feature_names = df.drop(columns='income').columns.tolist()
categorical_features = [feature_names.index(col) for col in categorical_columns]
continuous_features = [i for i in range(n_features) if i not in categorical_features]

categorical_names = {}
for feature in categorical_features:
    le_feat = LabelEncoder()
    le_feat.fit(data_raw[:, feature])
    data_raw[:, feature] = le_feat.transform(data_raw[:, feature])
    categorical_names[feature] = le_feat.classes_

data = data_raw.astype(float)
variances = np.zeros(data.shape[1])
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

encoder = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough"
)

train, test, labels_train, labels_test = train_test_split(data, labels, train_size=0.80, stratify=labels,random_state=42)
encoder.fit(data)
encoded_train = encoder.transform(train)
encoded_test = encoder.transform(test)

model = xgb.XGBClassifier(n_estimators=300, max_depth=10,random_state=42)
model.fit(encoded_train, labels_train)

instance = test[0]
acc = sklearn.metrics.accuracy_score(labels_test, model.predict(encoded_test))
print("Accuracy:", acc)

beta_init=np.ones(len(instance))#
predict_fn = lambda x: model.predict_proba(encoder.transform(x))


In [ ]:

from mixed_data_pl_lime import LocalModelStabilityAnalyzer
results_xlime = {}
rng = np.random.default_rng(4)
idx = rng.choice(len(test), size=50, replace=False)

for j in idx:
    analyzer = LocalModelStabilityAnalyzer(
        predict_fn, train, labels_train, test[j],
        categorical_features=categorical_features,
        continuous_features=continuous_features,
        groupk=range(1,X.shape[1]+1), n0=2, N=50, n_perturbation=6250,
        scale_factor=1, save_path=None
    )
    re = analyzer.run_analysis(run_name=f"inst_{j}")
    results_xlime[j] = re

In [ ]:

from mixed_data_lime import LocalModelStabilityAnalyzer
rng = np.random.default_rng(4)
idx = rng.choice(len(test), size=50, replace=False)
results_lime = {}

for j in [0]:
    analyzer1 = LocalModelStabilityAnalyzer(predict_fn, train, labels_train, test[j],categorical_features=categorical_features,continuous_features=continuous_features,groupk=range(1,X.shape[1]+1), N=50, n_perturbation=6250,scale_factor=1,save_path=None)

    re1=analyzer1.run_analysis(run_name=f"save_number{j}")
    results_lime[j] = re1
